In [1]:
! pip install langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.2.9-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached uuid_utils-0.14.0-cp39-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached orjson-3.11.7-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_community.document_loaders import SQLDatabaseLoader
from langchain_community.utilities import SQLDatabase
import os
# 1. إعداد الاتصال بقاعدة البيانات (مثلاً SQLite)
# لو عندك MySQL أو PostgreSQL بتغير الـ URI بس
db_path = r"C:/Users/Lenovo/Desktop/rag/sales.db"

# 2. اتأكد إن الملف موجود فعلاً قبل ما تبدأ
if os.path.exists(db_path):
    print(f"✅ تم العثور على الملف! حجمه: {os.path.getsize(db_path)} bytes")
    
    # ربط قاعدة البيانات
    db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
# 2. تحديد الاستعلام (Query) اللي هيجيب البيانات
# 1. التعديل الأهم: هات كل العواميد باستخدام النجمة *
# 1. غير الأسامي هنا باللي طلعلك من كود الكشف
query = "SELECT * FROM products" 

# 2. عدل الـ Loader
loader = SQLDatabaseLoader(
        query=query, 
        db=db, 
        # هنستخدم الطريقة اللي بتلم كل العواميد عشان نضمن إن مفيش أخطاء أسامي
        page_content_mapper=lambda row: "\n".join([f"{k}: {v}" for k, v in row.items()])
    )

db_documents = loader.load()

print("مبروك! البيانات اتحملت بنجاح.")
print(f"عدد الصفوف اللي اتحملت: {len(db_documents)}")
if len(db_documents) > 0:
    print("محتوى أول صف:")
    print(db_documents[0].page_content)
    print("-" * 30)
    print("المعلومات الإضافية (Metadata):")
    print(db_documents[0].metadata)
else:
    print("مفيش داتا عشان تتعرض، القائمة فاضية!")

✅ تم العثور على الملف! حجمه: 36864 bytes
مبروك! البيانات اتحملت بنجاح.
عدد الصفوف اللي اتحملت: 500
محتوى أول صف:
ProductID: 1
ProductName: Pesto Genovese
SupplierID: 25
CategoryID: 2
QuantityPerUnit: 12 - 150 g jars
UnitPrice: 25.0
UnitsInStock: 40
UnitsOnOrder: 0
ReorderLevel: 5
Discontinued: n
------------------------------
المعلومات الإضافية (Metadata):
{}


In [ ]:
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document

class DBEmbeddingPipeline:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        print(f"[INFO] Loaded embedding model: {model_name}")

    def fetch_data_from_db(self, db_path: str):
        # الاتصال بقاعدة البيانات
        conn = sqlite3.connect(db_path)
        # دي بتخلي النتيجة ترجع كـ Dictionary بدل Tuple عشان نعرف أسماء الأعمدة
        conn.row_factory = sqlite3.Row 
        cursor = conn.cursor()
        
        # سحب كل الصفوف وكل الأعمدة
        query = "SELECT * FROM products"
        cursor.execute(query)
        rows = cursor.fetchall()
        
        documents = []
        for row in rows:
            # تحويل الصف بالكامل لنص (ColumnName: Value)
            # بندمج كل الأعمدة في نص واحد للـ page_content
            content_parts = [f"{key}: {row[key]}" for key in row.keys()]
            full_content = " | ".join(content_parts)
            
            # تخزين البيانات: المحتوى فيه كل شيء، والـ Metadata فيها نسخة أصلية كـ dict
            doc = Document(
                page_content=full_content,
                metadata=dict(row) # بيحفظ الصف كله كـ dictionary في الميتا داتا
            )
            documents.append(doc)
            
        conn.close()
        print(f"[INFO] Fetched {len(documents)} products with ALL columns.")
        return documents

    def embed_documents(self, documents):
        texts = [doc.page_content for doc in documents]
        print(f"[INFO] Generating embeddings for {len(texts)} products...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

# --- التشغيل ---
if __name__ == "__main__":
    db_path = "sales.db" 
    pipeline = DBEmbeddingPipeline()
    
    
    db_docs = pipeline.fetch_data_from_db(db_path)
    
    
    db_embeddings = pipeline.embed_documents(db_docs)
    
    # تجربة: طباعة أول منتج عشان تشوف شكله بقا عامل ازاي
    print("\n--- Example Document Structure ---")
    print("Content:", db_docs[0].page_content) # هتلاقي فيه كل الأعمدة
    print("Metadata:", db_docs[0].metadata)     # هتلاقي فيه الـ ID وكل حاجة تانية

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1033.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Fetched 500 products with ALL columns.
[INFO] Generating embeddings for 500 products...


Batches: 100%|██████████| 16/16 [00:01<00:00,  8.83it/s]


--- Example Document Structure ---
Content: ProductID: 1 | ProductName: Pesto Genovese | SupplierID: 25 | CategoryID: 2 | QuantityPerUnit: 12 - 150 g jars | UnitPrice: 25.0 | UnitsInStock: 40 | UnitsOnOrder: 0 | ReorderLevel: 5 | Discontinued: n
Metadata: {'ProductID': 1, 'ProductName': 'Pesto Genovese', 'SupplierID': 25, 'CategoryID': 2, 'QuantityPerUnit': '12 - 150 g jars', 'UnitPrice': 25.0, 'UnitsInStock': 40, 'UnitsOnOrder': 0, 'ReorderLevel': 5, 'Discontinued': 'n'}


In [25]:
# --- 1. تعريف الكلاسات (تأكد من تشغيل هذا الجزء أولاً) ---
import os
import faiss
import numpy as np
import pickle
import sqlite3
from typing import List, Any
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import SQLDatabaseLoader
from langchain_community.utilities import SQLDatabase
from sentence_transformers import CrossEncoder

class EmbeddingPipeline:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)

    def chunk_documents(self, documents: List[Document]) -> List[Document]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap
        )
        return splitter.split_documents(documents)

    def embed_chunks(self, chunks: List[Document]) -> np.ndarray:
        texts = [chunk.page_content for chunk in chunks]
        return self.model.encode(texts, show_progress_bar=True)

class FaissVectorStore:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2"):
        self.persist_dir = persist_dir
        os.makedirs(self.persist_dir, exist_ok=True)
        self.index = None
        self.metadata = []
        self.model = SentenceTransformer(embedding_model)
        print("[INFO] Loading Reranker model (Cross-Encoder)...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-base')

    def build_from_documents(self, documents: List[Document]):
        # هنا بنادي على الـ Pipeline من جوه الكلاس
        emb_pipe = EmbeddingPipeline(model_name="all-MiniLM-L6-v2")
        chunks = emb_pipe.chunk_documents(documents)
        embeddings = emb_pipe.embed_chunks(chunks)
        
        # حفظ النص بالكامل في الميتا داتا عشان يظهر في نتائج البحث
        metadatas = [{"text": chunk.page_content} for chunk in chunks]
        
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(np.array(embeddings).astype('float32'))
        self.metadata.extend(metadatas)
        self.save()

    def save(self):
        faiss.write_index(self.index, os.path.join(self.persist_dir, "faiss.index"))
        with open(os.path.join(self.persist_dir, "metadata.pkl"), "wb") as f:
            pickle.dump(self.metadata, f)

    def load(self):
        self.index = faiss.read_index(os.path.join(self.persist_dir, "faiss.index"))
        with open(os.path.join(self.persist_dir, "metadata.pkl"), "rb") as f:
            self.metadata = pickle.load(f)

    def query(self, query_text: str, top_k: int = 5):
        query_emb = self.model.encode([query_text]).astype('float32')
        D, I = self.index.search(query_emb, top_k)
        results = []
        for idx, dist in zip(I[0], D[0]):
            if idx < len(self.metadata):
                results.append({"text": self.metadata[idx]["text"], "distance": dist})
        return results
    def query_with_rerank(self, query_text: str, top_k_initial: int = 20, top_k_final: int = 5):
        # الخطوة 1: جلب نتائج أولية كتيرة شوية من FAISS
        query_emb = self.model.encode([query_text]).astype('float32')
        D, I = self.index.search(query_emb, top_k_initial)
        
        initial_results = []
        for idx in I[0]:
            if idx < len(self.metadata):
                initial_results.append(self.metadata[idx]["text"])
        
        if not initial_results:
            return []

        # الخطوة 2: الـ Reranking
        # بنعمل أزواج من (السؤال، النتيجة) للموديل
        pairs = [[query_text, res] for res in initial_results]
        scores = self.reranker.predict(pairs)
        
        # الخطوة 3: إعادة الترتيب بناءً على الـ Scores الجديدة
        reranked_results = sorted(
            zip(initial_results, scores), 
            key=lambda x: x[1], 
            reverse=True
        )

        # إرجاع أفضل النتائج بعد الترتيب
        return reranked_results[:top_k_final]

# --- تجربة البحث مع Rerank ---
if os.path.exists(db_path):
    # (تأكد إنك عملت load للـ store الأول)
    store = FaissVectorStore(persist_dir="db_faiss_index")
    store.load()
    
    query = "Milk Semi-Skimmed 1L"
    print(f"\n[INFO] Searching for: {query} with Reranking...")
    
    final_results = store.query_with_rerank(query)
    
    for text, score in final_results:
        print(f"Score: {score:.4f} | Result: {text}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 581.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Loading Reranker model (Cross-Encoder)...


c:\Users\Lenovo\Desktop\rag\rrr\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1083.43it/s, Materializing param=roberta.encoder.layer.11


[INFO] Searching for: Milk Semi-Skimmed 1L with Reranking...
Score: 0.9925 | Result: ProductID: 437 | ProductName: Milk Semi-Skimmed 1L | SupplierID: 17 | CategoryID: 4 | QuantityPerUnit: 1L carton | UnitPrice: 2.5 | UnitsInStock: 250 | UnitsOnOrder: 0 | ReorderLevel: 50 | Discontinued: n
Score: 0.0908 | Result: ProductID: 436 | ProductName: Milk Whole 1L | SupplierID: 17 | CategoryID: 4 | QuantityPerUnit: 1L carton | UnitPrice: 2.5 | UnitsInStock: 200 | UnitsOnOrder: 0 | ReorderLevel: 50 | Discontinued: n
Score: 0.0886 | Result: ProductID: 177 | ProductName: Oat Milk Barista Edition | SupplierID: 10 | CategoryID: 1 | QuantityPerUnit: 1L carton | UnitPrice: 5.5 | UnitsInStock: 120 | UnitsOnOrder: 40 | ReorderLevel: 25 | Discontinued: n
Score: 0.0782 | Result: ProductID: 266 | ProductName: Buttermilk Fresh | SupplierID: 17 | CategoryID: 4 | QuantityPerUnit: 1L carton | UnitPrice: 4.5 | UnitsInStock: 80 | UnitsOnOrder: 0 | ReorderLevel: 20 | Discontinued: n
Score: 0.0599 | Result: Produ

In [26]:
def evaluate_retrieval_accuracy(store, db_documents, num_tests=50):
    
    import random
    
    hits = 0
    total_tests = min(num_tests, len(db_documents))
    
    # اختيار عينة عشوائية للاختبار
    test_samples = random.sample(db_documents, total_tests)
    
    print(f"[INFO] Starting evaluation on {total_tests} samples...")
    
    for doc in test_samples:
        # استخراج اسم المنتج من الـ metadata أو المحتوى ككلمة بحث
        # بنفترض إن اسم المنتج موجود في الميتا داتا اللي خزنناها
        actual_name = doc.metadata.get('ProductName') 
        
        # لو مش موجود في الميتا داتا، هنحاول نجيبه من النص (أول جزء قبل الـ '|')
        if not actual_name:
            actual_name = doc.page_content.split('|')[0].split(':')[1].strip()

        # عمل بحث باستخدام اسم المنتج
        results = store.query(actual_name, top_k=3) # بنشوف هل هو في أول 3 نتائج؟
        
        # التأكد إذا كان المنتج الصحيح موجود في النتائج
        retrieved_texts = [r['text'] for r in results]
        
        for text in retrieved_texts:
            if actual_name in text:
                hits += 1
                break
                
    accuracy = (hits / total_tests) * 100
    print(f"\n--- Evaluation Results ---")
    print(f"Total Queries: {total_tests}")
    print(f"Successful Hits: {hits}")
    print(f"Accuracy (Hit Rate @ 3): {accuracy:.2f}%")
    return accuracy

# --- تشغيل التقييم ---
# بعد ما تكون عملت الـ store والـ db_documents
accuracy = evaluate_retrieval_accuracy(store, db_documents, num_tests=100)

[INFO] Starting evaluation on 100 samples...

--- Evaluation Results ---
Total Queries: 100
Successful Hits: 18
Accuracy (Hit Rate @ 3): 18.00%
